# Data preparation

Tạo toàn bộ dữ liệu benchmark cho FPClose bằng Julia:

1. Tải 9 dataset gốc (SPMF, FIMI) và chuẩn hóa sang định dạng SPMF.
2. Thống kê 9 dataset.
3. Sinh 5 Retail subsets (thí nghiệm scalability).
4. Sinh 6 synthetic datasets (thí nghiệm độ dài giao dịch).

Mỗi bước kiểm tra SHA-256 với giá trị trong `data/benchmark/MANIFEST.md`, nên dữ liệu tạo ra
giống hệt dữ liệu nhóm dùng.

**Cách chạy:** mở notebook với kernel Julia (VS Code, Jupyter hoặc Colab với runtime Julia) rồi *Run all*,
hoặc chạy lần lượt từng cell. Chạy trong repo thì dữ liệu được ghi vào `data/`; chạy ở nơi khác thì ghi
vào thư mục `fim_data/`. Bước 1 tải khoảng 150 MB.

## 0 · Thiết lập thư mục và hồ sơ của 9 dataset

Xác định nơi ghi dữ liệu, và ghi ra hai bảng hồ sơ (nguồn tải, SHA-256, số giao dịch) dùng để đối chiếu ở các bước sau.

In [ ]:
# Thư mục gốc: repo FPClose nếu đang ở trong repo, ngược lại là ./fim_data
function find_repo(dir = pwd())
    while true
        isfile(joinpath(dir, "data", "benchmark", "MANIFEST.md")) && return dir
        parent = dirname(dir)
        parent == dir && return nothing
        dir = parent
    end
end
REPO = find_repo()
ROOT = REPO === nothing ? joinpath(pwd(), "fim_data") : REPO
DATA = joinpath(ROOT, "data")
RAW_DIR = joinpath(DATA, "raw")
BENCHMARK_DIR = joinpath(DATA, "benchmark")
SCALABILITY_DIR = joinpath(DATA, "scalability")
SYNTHETIC_DIR = joinpath(DATA, "synthetic")
foreach(mkpath, (RAW_DIR, BENCHMARK_DIR, SCALABILITY_DIR, SYNTHETIC_DIR))

# Nguồn, SHA-256 và thống kê của 9 dataset (data/results/dataset_statistics.csv)
CONFIG_DIR = mkpath(joinpath(tempdir(), "fim_setup"))
STATISTICS_CSV = joinpath(CONFIG_DIR, "dataset_statistics.csv")
write(STATISTICS_CSV, raw"""
dataset,n_transactions,n_unique_items,avg_transaction_length,max_transaction_length,density_pct,reference_n_transactions,reference_count_matches,source_url,raw_filename,raw_sha256,clean_sha256,id_offset
chess,3196,75,37.0,37,49.333333,3196,True,https://www.philippe-fournier-viger.com/spmf/publicdatasets/chess.txt,chess.txt,a12ea887df58a396709430af5bf0a9a32d1f6eba8e7c13dd41f28b98572c5db2,341d0ff8676e053dd3b417d61f84c732736154f987ce56975a099b07460957f5,unknown_original_not_logged
mushrooms,8416,119,23.0,23,19.327731,8416,True,https://www.philippe-fournier-viger.com/spmf/publicdatasets/mushrooms.txt,mushrooms.txt,3bc1159b06baa231932810d1d58cbb958684be88b6daa8e3672f8d76c5f0eb98,37a11571574d74bd1659c830a1e30e120485c6994b79f32e3b6635621a229bf7,unknown_original_not_logged
retail,88162,16470,10.3058,76,0.062573,88162,True,https://www.philippe-fournier-viger.com/spmf/publicdatasets/retail.txt,retail.txt,732c26de19888cb570d3fbb97e47206a9b1c0ce064f85dc1403314ba38f04eaa,417563fb5feb3711d4f761230ca78b76d100fe2ee0d3178fcc4fbb000d8d1c36,unknown_original_not_logged
T10I4D100K,100000,870,10.1023,29,1.161182,100000,True,https://www.philippe-fournier-viger.com/spmf/publicdatasets/T10I4D100K.txt,T10I4D100K.txt,949983f61aa976d1b774474fadd31c5f15775b24728a5e7c8b79801822ab3f89,fb2dd118ff822ff88d724e47d5a4cea6531c0bd6294397db4d36890c1450f1a7,unknown_original_not_logged
accidents,340183,468,33.8079,51,7.223908,340183,True,https://www.philippe-fournier-viger.com/spmf/publicdatasets/accidents.txt,accidents.txt,8c2275550966e3cb4b6066f2bb168489a9566c500af559fa3925f1b11d97f0cd,c2b3762e3f8f0ca241c35359b24e3015fd34ea241d09f94b42ae44ed42914726,unknown_original_not_logged
connect,67557,129,43.0,43,33.333333,67557,True,https://www.philippe-fournier-viger.com/spmf/publicdatasets/connect.txt,connect.txt,7a23cb64573f12b6d967bdc44a81d6acdb89b00edf2ff46cc970ead6cc67b8e2,650a92d7803c465fb08fc0e58de087ceab84cd5f0de4c6816b523774dcd6a2c9,0
kosarak,990002,41270,8.1,2497,0.019627,990002,True,https://cs.rpi.edu/~zaki/Workshops/FIMI/data/kosarak.dat.gz,kosarak.dat.gz,b7f575183ba6e7f61e9a7b51ec386d33ce503a14227832ad5986bb57baaf0102,cb4270629d786207143bb3c4002c36aedc3824f55294c7b3d5538c8023cec314,0
pumsb_star,49046,2088,50.4821,63,2.417727,49046,True,https://cs.rpi.edu/~zaki/Workshops/FIMI/data/pumsb_star.dat.gz,pumsb_star.dat.gz,7386d5bd0b56611942a365958003430478d3d47cd920199736b602776005aada,a9ecd5c882ffff9e0042703253d0eef3453f747537e0027c9be275a8aaf38f2f,1
T40I10D100K,100000,942,39.6051,77,4.20436,100000,True,https://cs.rpi.edu/~zaki/Workshops/FIMI/data/T40I10D100K.dat,T40I10D100K.dat,3939b939cba8380de51a00e9831d975639678f7b4115e4a6f29ca50d3d695339,e3d86e8cf6eec0093528e238ab8a4ba10471375c00a9298144edb462caadb28c,1
""")

# SHA-256 của Retail subsets và synthetic datasets (data/results/derived_hashes.csv)
DERIVED_CSV = joinpath(CONFIG_DIR, "derived_hashes.csv")
write(DERIVED_CSV, raw"""
dataset,group,n_transactions,sha256
retail_p010.txt,scalability,8816,4e806f7b92ad4c165e1d43c199f071d8c967cd20c7ea8c0cf392cb9aab2d54bf
retail_p025.txt,scalability,22040,852d2ef02eb8e83fafc53c9a020ddce082ccc99bc8417f293ee63a9c513bd100
retail_p050.txt,scalability,44081,6c6ee4405939af858fa934473794f32474a43408088d10be8f15561290bb7662
retail_p075.txt,scalability,66121,af001d2e72523f52a66afd03a054d81dbd8620e6f3e107ecab00ec2124fd371a
retail_p100.txt,scalability,88162,105bcad96d35013ee31d5d4a2746b7a36b843347f35c4e213cb5fcb94af41068
len05.txt,synthetic,10000,351794615fadd2f8a2acdda7cea536a17ab2fbcfb3c56fd2a404a972fe6e0934
len10.txt,synthetic,10000,2bcf15fdc58fea9ebc1c3d8cf2246b3d132a6c534ccb2e8bec095adb55ee9fba
len15.txt,synthetic,10000,1e2cd059363ea3c14c390b646c470b62ad16c19dc8ae3e563a57adb0de8217e5
len20.txt,synthetic,10000,e3f051d4c8b86e79797c9d1732ce07f9add9257deae2a69b7dbbe9961349af71
len25.txt,synthetic,10000,795927642abb1486aee60bb1a6e8d0c6fbb023751bc60b45d74c5e2bb3976ee8
len30.txt,synthetic,10000,5b16994308bf4dc443efc3720467be0bab220aef499356706c463077f7c535d9
""")

println("Julia ", VERSION)
println("Dữ liệu sẽ ghi vào: ", DATA)

### Hàm dùng chung: tính SHA-256, đọc và ghi CSV

Dùng ở mọi bước bên dưới để kiểm tra file sinh ra có đúng từng byte với bản nhóm đã công bố hay không.

Hàm dùng chung cho các script chuẩn bị dữ liệu benchmark.
Chỉ dùng thư viện chuẩn của Julia (SHA), không cần thêm gói nào.

In [ ]:
using SHA

"""
    sha256_file(path) -> String

SHA-256 (hex, chữ thường) của file, đọc theo luồng nên không tốn RAM với file lớn.
"""
sha256_file(path::AbstractString) = open(io -> bytes2hex(sha256(io)), path)

"""
    read_csv_rows(path) -> Vector{Dict{String,String}}

Đọc CSV đơn giản (không có dấu phẩy trong ô), đủ cho các file trong `data/results/`.
"""
function read_csv_rows(path::AbstractString)
    lines = filter(!isempty, strip.(readlines(path)))
    header = split(lines[1], ',')
    return [Dict(String(h) => String(v) for (h, v) in zip(header, split(l, ','))) for l in lines[2:end]]
end

"""
    write_csv(path, header, rows)

Ghi CSV với xuống dòng LF. `rows` là danh sách vector giá trị theo đúng thứ tự `header`.
"""
function write_csv(path::AbstractString, header::Vector{String}, rows)
    mkpath(dirname(abspath(path)))
    open(path, "w") do io
        println(io, join(header, ','))
        for r in rows
            println(io, join(string.(r), ','))
        end
    end
end

## 1 · Tải và chuẩn hóa 9 dataset

### Đọc file gốc (kể cả file nén gzip) và chuẩn hóa sang định dạng SPMF

Chuẩn hóa một file giao dịch thô (SPMF/FIMI) về định dạng chuẩn của nhóm.

Quy tắc (giống MANIFEST.md):
- Nhận diện gzip theo magic bytes 1f 8b, KHÔNG theo đuôi file
  (T40I10D100K.dat tải từ FIMI thực chất là gzip).
- Bỏ BOM UTF-8 và các dòng metadata bắt đầu bằng #, @, %.
- Dòng trống hoặc item ≤ 0 (sau khi cộng offset) thì báo lỗi, không tự bỏ qua.
- Cộng offset vào mọi item ID (1 khi nguồn có ID 0), khử item trùng, sắp tăng dần.
- Ghi UTF-8, mỗi giao dịch một dòng, xuống dòng LF.

Offset đã dùng: 0 cho chess, mushrooms, retail, T10I4D100K, accidents, connect, kosarak;
1 cho pumsb_star, T40I10D100K.

In [ ]:
using Zlib_jll: libz

"File có phải gzip không (2 byte đầu là 1f 8b)."
is_gzip(path::AbstractString) = open(io -> read(io, 2) == UInt8[0x1f, 0x8b], path)

"""
    gunzip(path) -> IOBuffer

Giải nén file gzip bằng zlib đi kèm Julia (`Zlib_jll`), không cần lệnh `gzip` của hệ điều hành
(Windows thường không có).
"""
function gunzip(path::AbstractString)
    gz = ccall((:gzopen, libz), Ptr{Cvoid}, (Cstring, Cstring), path, "rb")
    gz == C_NULL && error("Không mở được file gzip: $path")
    out = IOBuffer()
    buf = Vector{UInt8}(undef, 1 << 20)
    try
        while true
            n = ccall((:gzread, libz), Cint, (Ptr{Cvoid}, Ptr{UInt8}, Cuint), gz, buf, length(buf))
            n < 0 && error("Lỗi giải nén gzip: $path")
            n == 0 && break
            write(out, view(buf, 1:n))
        end
    finally
        ccall((:gzclose, libz), Cint, (Ptr{Cvoid},), gz)
    end
    return seekstart(out)
end

"Mở file thô để đọc text; file gzip (nhận theo magic bytes) được giải nén trước."
open_raw(path::AbstractString) = is_gzip(path) ? gunzip(path) : open(path)

"""
    normalize_transactions(src, dst; offset = 0) -> (n_transactions, n_dedup_rows, n_metadata)

Chuẩn hóa `src` vào `dst`. Trả về số giao dịch ghi ra, số dòng có item trùng đã khử, số dòng metadata đã bỏ.
"""
function normalize_transactions(src::AbstractString, dst::AbstractString; offset::Integer = 0)
    mkpath(dirname(abspath(dst)))
    count = 0
    dedup_rows = 0
    metadata = 0
    input = open_raw(src)
    try
        open(dst, "w") do out
            for (line_number, raw) in enumerate(eachline(input))
                line = line_number == 1 ? lstrip(raw, '﻿') : raw
                line = strip(line)
                if startswith(line, '#') || startswith(line, '@') || startswith(line, '%')
                    metadata += 1
                    continue
                end
                isempty(line) && error("Giao dịch trống tại $src:$line_number")
                values = parse.(Int, split(line))
                minimum(values) + offset > 0 || error("Item ID không hợp lệ tại $src:$line_number")
                items = sort!(unique!(values .+ offset))
                length(items) < length(values) && (dedup_rows += 1)
                println(out, join(items, ' '))
                count += 1
            end
        end
    finally
        close(input)
    end
    return (n_transactions = count, n_dedup_rows = dedup_rows, n_metadata = metadata)
end

### Tải file gốc, kiểm tra SHA-256 rồi chuẩn hóa

Tải 9 dataset gốc từ SPMF/FIMI rồi chuẩn hóa, so với hồ sơ đã lưu.

Đọc URL, tên file raw, SHA-256 raw/clean và số giao dịch từ
data/results/dataset_statistics.csv. Với mỗi dataset:
1. tải raw vào thư mục raw (bỏ qua nếu đã có và đúng hash), so SHA-256 raw;
2. chuẩn hóa bằng `normalize_transactions` với offset trong DATASET_OFFSETS;
3. so số giao dịch và SHA-256 bản chuẩn hóa với hồ sơ.

Raw khác hash (nguồn đã đổi) vẫn được chuẩn hóa nhưng báo rõ để biết nguyên nhân.

In [ ]:
using Downloads

"Item-ID offset đã dùng cho từng dataset (xem MANIFEST.md)."
const DATASET_OFFSETS = Dict(
    "chess" => 0, "mushrooms" => 0, "retail" => 0, "T10I4D100K" => 0, "accidents" => 0,
    "connect" => 0, "kosarak" => 0, "pumsb_star" => 1, "T40I10D100K" => 1)

"""
    download_and_normalize(statistics, rawdir, outdir; only = nothing) -> Vector{NamedTuple}

Trả về một dòng kết quả cho mỗi dataset: raw_ok, clean_ok, số giao dịch, số dòng đã khử item trùng.
"""
function download_and_normalize(statistics::AbstractString, rawdir::AbstractString,
        outdir::AbstractString; only = nothing)
    mkpath(rawdir)
    mkpath(outdir)
    results = NamedTuple[]
    for row in read_csv_rows(statistics)
        name = row["dataset"]
        only === nothing || name in only || continue
        raw = joinpath(rawdir, row["raw_filename"])
        raw_expected = lowercase(row["raw_sha256"])
        if !(isfile(raw) && sha256_file(raw) == raw_expected)
            println("Tải ", row["source_url"])
            Downloads.download(row["source_url"], raw)
        end
        raw_ok = sha256_file(raw) == raw_expected

        dst = joinpath(outdir, "$name.txt")
        r = normalize_transactions(raw, dst; offset = DATASET_OFFSETS[name])
        clean = sha256_file(dst)
        clean_ok = clean == lowercase(row["clean_sha256"]) &&
                   r.n_transactions == parse(Int, row["n_transactions"])
        push!(results, (dataset = name, raw_ok = raw_ok, gzip = is_gzip(raw),
            offset = DATASET_OFFSETS[name], n_transactions = r.n_transactions,
            dedup_rows = r.n_dedup_rows, metadata_removed = r.n_metadata,
            clean_ok = clean_ok, clean_sha256 = clean))
        println(clean_ok ? "[PASS] " : "[FAIL] ", rpad(name, 12),
            "raw=", raw_ok ? "khớp" : "KHÁC", "  gzip=", is_gzip(raw),
            "  offset=", DATASET_OFFSETS[name], "  n=", r.n_transactions,
            "  dedup_rows=", r.n_dedup_rows)
    end
    return results
end

### Chạy bước 1

File raw đã có và đúng hash thì không tải lại.

In [ ]:
benchmark = download_and_normalize(STATISTICS_CSV, RAW_DIR, BENCHMARK_DIR)
@assert all(r -> r.clean_ok, benchmark) "Có dataset không khớp SHA-256 trong MANIFEST"
println("OK: 9/9 dataset đã chuẩn hóa trong ", BENCHMARK_DIR)

## 2 · Thống kê 9 dataset

### Tính số giao dịch, số item, độ dài giao dịch và mật độ của một dataset

Thống kê cho các file giao dịch đã chuẩn hóa.

Cột: dataset, n_transactions, n_unique_items, avg_transaction_length,
max_transaction_length, density_pct, sha256.
density_pct = avg_transaction_length / n_unique_items × 100 (đơn vị %, khác cột
`density` dạng tỉ lệ trong synthetic_statistics.csv).

In [ ]:
const STATS_HEADER = ["dataset", "n_transactions", "n_unique_items", "avg_transaction_length",
    "max_transaction_length", "density_pct", "sha256"]

"Thống kê một file; đọc từng dòng nên dùng được cả với kosarak (~1 triệu giao dịch)."
function dataset_stats(path::AbstractString)
    n = 0
    total = 0
    maxlen = 0
    items = Set{Int}()
    for line in eachline(path)
        s = strip(line)
        isempty(s) && error("Dòng trống trong $path")
        k = 0
        for tok in eachsplit(s)
            push!(items, parse(Int, tok))
            k += 1
        end
        n += 1
        total += k
        maxlen = max(maxlen, k)
    end
    avg = total / n
    return (dataset = splitext(basename(path))[1], n_transactions = n,
        n_unique_items = length(items), avg_transaction_length = round(avg; digits = 4),
        max_transaction_length = maxlen,
        density_pct = round(avg / length(items) * 100; digits = 6), sha256 = sha256_file(path))
end

stats_row(s) = [getfield(s, Symbol(h)) for h in STATS_HEADER]

### Chạy bước 2

In [ ]:
println(join(STATS_HEADER, ','))
for f in sort(filter(f -> endswith(f, ".txt"), readdir(BENCHMARK_DIR; join = true)))
    println(join(string.(stats_row(dataset_stats(f))), ','))
end

## 3 · Retail subsets

### Bộ sinh số ngẫu nhiên giống hệt `random` của Python

Dùng cho cả bước 3 và bước 4.

Cài lại bằng Julia đúng bộ sinh số ngẫu nhiên `random.Random` của CPython.

Vì sao cần: Retail subsets và synthetic datasets đã công bố được sinh bằng
`random.Random(2026)` của Python. `MersenneTwister(2026)` của Julia cho dãy số
khác, nên muốn tái tạo đúng từng byte (cùng SHA-256) phải chép lại:
- MT19937 và cách CPython khởi tạo seed số nguyên (init_by_array),
- random(), getrandbits(k), _randbelow(n),
- shuffle, randint, sample (trên range), choices (có weights).

Thuật toán theo mã nguồn CPython 3.x (Modules/_randommodule.c, Lib/random.py).

In [ ]:
const _MT_N = 624
const _MT_M = 397
const _MATRIX_A = 0x9908b0df
const _UPPER_MASK = 0x80000000
const _LOWER_MASK = 0x7fffffff

"""
    PyRandom(seed)

Tương đương `random.Random(seed)` của Python với `seed` là số nguyên.
"""
mutable struct PyRandom
    mt::Vector{UInt32}
    mti::Int
end

function PyRandom(seed::Integer)
    # CPython: seed nguyên -> abs(seed) -> mảng các từ 32 bit, little-endian.
    s = abs(BigInt(seed))
    key = UInt32[]
    if s == 0
        push!(key, UInt32(0))
    else
        while s > 0
            push!(key, UInt32(s & 0xffffffff))
            s >>= 32
        end
    end

    # init_genrand(19650218)
    mt = zeros(UInt32, _MT_N)
    mt[1] = UInt32(19650218)
    for i in 1:_MT_N-1
        prev = mt[i]
        mt[i+1] = UInt32(1812433253) * (prev ⊻ (prev >> 30)) + UInt32(i)
    end

    # init_by_array(key). i, j là chỉ số 0-based như mã C; mt[i+1] là mt[i] của C.
    i, j = 1, 0
    klen = length(key)
    for _ in 1:max(_MT_N, klen)
        prev = mt[i]
        mt[i+1] = (mt[i+1] ⊻ ((prev ⊻ (prev >> 30)) * UInt32(1664525))) + key[j+1] + UInt32(j)
        i += 1
        j += 1
        if i >= _MT_N
            mt[1] = mt[_MT_N]
            i = 1
        end
        j >= klen && (j = 0)
    end
    for _ in 1:_MT_N-1
        prev = mt[i]
        mt[i+1] = (mt[i+1] ⊻ ((prev ⊻ (prev >> 30)) * UInt32(1566083941))) - UInt32(i)
        i += 1
        if i >= _MT_N
            mt[1] = mt[_MT_N]
            i = 1
        end
    end
    mt[1] = 0x80000000
    return PyRandom(mt, _MT_N)
end

@inline _mag01(y::UInt32) = (y & 0x00000001) == 0 ? UInt32(0) : _MATRIX_A

"genrand_uint32 của CPython: một số nguyên 32 bit."
function genrand_uint32!(r::PyRandom)
    mt = r.mt
    if r.mti >= _MT_N
        for kk in 0:_MT_N-_MT_M-1
            y = (mt[kk+1] & _UPPER_MASK) | (mt[kk+2] & _LOWER_MASK)
            mt[kk+1] = mt[kk+_MT_M+1] ⊻ (y >> 1) ⊻ _mag01(y)
        end
        for kk in _MT_N-_MT_M:_MT_N-2
            y = (mt[kk+1] & _UPPER_MASK) | (mt[kk+2] & _LOWER_MASK)
            mt[kk+1] = mt[kk+(_MT_M-_MT_N)+1] ⊻ (y >> 1) ⊻ _mag01(y)
        end
        y = (mt[_MT_N] & _UPPER_MASK) | (mt[1] & _LOWER_MASK)
        mt[_MT_N] = mt[_MT_M] ⊻ (y >> 1) ⊻ _mag01(y)
        r.mti = 0
    end
    y = mt[r.mti+1]
    r.mti += 1
    y ⊻= y >> 11
    y ⊻= (y << 7) & 0x9d2c5680
    y ⊻= (y << 15) & 0xefc60000
    y ⊻= y >> 18
    return y
end

"`random.random()`: số thực trong [0, 1) với 53 bit ngẫu nhiên."
function pyrandom!(r::PyRandom)
    a = genrand_uint32!(r) >> 5
    b = genrand_uint32!(r) >> 6
    return (Float64(a) * 67108864.0 + Float64(b)) * (1.0 / 9007199254740992.0)
end

"`getrandbits(k)` với 1 ≤ k ≤ 32 (đủ cho mọi n dùng ở đây)."
function getrandbits!(r::PyRandom, k::Integer)
    1 <= k <= 32 || throw(ArgumentError("getrandbits chỉ hỗ trợ 1 ≤ k ≤ 32, nhận $k"))
    return Int(genrand_uint32!(r) >> (32 - k))
end

"`_randbelow(n)`: số nguyên đều trong [0, n), phương pháp loại bỏ như CPython."
function randbelow!(r::PyRandom, n::Integer)
    n > 0 || throw(ArgumentError("n phải dương"))
    k = ndigits(n; base = 2)          # n.bit_length()
    x = getrandbits!(r, k)
    while x >= n
        x = getrandbits!(r, k)
    end
    return x
end

"`random.shuffle(x)` (tại chỗ), cùng thứ tự gọi số ngẫu nhiên như Python."
function pyshuffle!(r::PyRandom, x::AbstractVector)
    for i in length(x)-1:-1:1         # i là chỉ số 0-based
        j = randbelow!(r, i + 1)
        x[i+1], x[j+1] = x[j+1], x[i+1]
    end
    return x
end

"`random.randint(a, b)`: số nguyên trong [a, b]."
pyrandint!(r::PyRandom, a::Integer, b::Integer) = a + randbelow!(r, b - a + 1)

"""
    pysample_range!(r, start, stop, k)

`random.sample(range(start, stop), k)`.
"""
function pysample_range!(r::PyRandom, start::Integer, stop::Integer, k::Integer)
    n = stop - start
    0 <= k <= n || throw(ArgumentError("k nằm ngoài [0, n]"))
    setsize = 21
    if k > 5
        setsize += 4^ceil(Int, log(3k) / log(4))
    end
    result = Vector{Int}(undef, k)
    if n <= setsize
        pool = collect(start:stop-1)
        for i in 0:k-1
            j = randbelow!(r, n - i)
            result[i+1] = pool[j+1]
            pool[j+1] = pool[n-i]
        end
    else
        selected = Set{Int}()
        for i in 0:k-1
            j = randbelow!(r, n)
            while j in selected
                j = randbelow!(r, n)
            end
            push!(selected, j)
            result[i+1] = start + j
        end
    end
    return result
end

"Cộng dồn trái sang phải như `itertools.accumulate`."
function cumulative(weights)
    cum = Vector{Float64}(undef, length(weights))
    acc = 0.0
    for (i, w) in enumerate(weights)
        acc += Float64(w)
        cum[i] = acc
    end
    return cum
end

"""
    pychoice_index!(r, cum_weights) -> Int

Một lần chọn của `random.choices(population, weights=..., k=1)`; trả về chỉ số 0-based
trong population. `cum_weights` lấy từ `cumulative(weights)`.
"""
function pychoice_index!(r::PyRandom, cum_weights::Vector{Float64})
    total = cum_weights[end] + 0.0
    x = pyrandom!(r) * total
    lo, hi = 0, length(cum_weights) - 1   # bisect_right(cum, x, 0, n-1)
    while lo < hi
        mid = (lo + hi) ÷ 2
        if x < cum_weights[mid+1]
            hi = mid
        else
            lo = mid + 1
        end
    end
    return lo
end

"""
    cpow(x, y)

`x ** y` của Python gọi hàm `pow` của thư viện C. Dùng cùng hàm đó (nếu gọi được) để
trọng số khớp từng bit với dữ liệu sinh trên Colab; nếu không thì dùng `^` của Julia.
"""
const cpow = try
    ccall(:pow, Float64, (Float64, Float64), 2.0, 0.5)
    (x::Float64, y::Float64) -> ccall(:pow, Float64, (Float64, Float64), x, y)
catch
    (x::Float64, y::Float64) -> x^y
end

### Chia Retail thành 5 tập con lồng nhau 10%, 25%, 50%, 75%, 100%

Sinh 5 Retail subsets lồng nhau cho thí nghiệm scalability.

Cách làm (giống bản đã công bố): xáo trộn CHỈ SỐ giao dịch một lần bằng
`random.Random(seed).shuffle` (bản port `PyRandom` ở trên), rồi lấy prefix
n*p÷100 phần tử cho p = 10, 25, 50, 75, 100. Vì cùng một hoán vị nên tập nhỏ
luôn nằm trong tập lớn (lồng nhau). Thứ tự dòng trong file theo thứ tự hoán vị.

In [ ]:
const RETAIL_PERCENTAGES = (10, 25, 50, 75, 100)
const RETAIL_N = 88162

"""
    make_retail_subsets(src, outdir; seed = 2026) -> Vector{NamedTuple}

Ghi `retail_p010.txt` … `retail_p100.txt` vào `outdir`, trả về tên file, số giao dịch, SHA-256.
"""
function make_retail_subsets(src::AbstractString, outdir::AbstractString; seed::Integer = 2026)
    transactions = String[]
    for line in eachline(src)
        s = strip(line)
        isempty(s) || push!(transactions, String(s))
    end
    length(transactions) == RETAIL_N ||
        error("retail.txt có $(length(transactions)) giao dịch, mong đợi $RETAIL_N")

    order = collect(0:length(transactions)-1)
    pyshuffle!(PyRandom(seed), order)

    mkpath(outdir)
    reports = NamedTuple[]
    previous = Set{Int}()
    for p in RETAIL_PERCENTAGES
        n = length(transactions) * p ÷ 100
        chosen = @view order[1:n]
        current = Set(chosen)
        issubset(previous, current) || error("Subset $p% không chứa subset trước")
        previous = current
        file = joinpath(outdir, "retail_p$(lpad(p, 3, '0')).txt")
        open(file, "w") do io
            for idx in chosen
                write(io, transactions[idx+1], '\n')
            end
        end
        push!(reports, (dataset = basename(file), n_transactions = n, sha256 = sha256_file(file)))
    end
    return reports
end

### Chạy bước 3

In [ ]:
expected = Dict(r["dataset"] => r["sha256"] for r in read_csv_rows(DERIVED_CSV))
subsets = make_retail_subsets(joinpath(BENCHMARK_DIR, "retail.txt"), SCALABILITY_DIR)
for r in subsets
    println(r.sha256 == expected[r.dataset] ? "[OK]   " : "[SAI]  ", rpad(r.dataset, 18), r.n_transactions)
end
@assert all(r -> r.sha256 == expected[r.dataset], subsets) "Retail subsets không khớp SHA-256"

## 4 · Synthetic datasets

### Sinh 6 dataset có độ dài giao dịch trung bình 5, 10, 15, 20, 25, 30

Sinh 6 synthetic datasets cho thí nghiệm ảnh hưởng của độ dài giao dịch.

Cơ chế (giống bản đã công bố, seed 2026):
- 500 item; 200 pattern 4 item: 125 pattern đầu chia đều toàn bộ 500 item
  (sau một lần xáo trộn), 75 pattern sau lấy ngẫu nhiên.
- Pattern được chọn theo trọng số 1/(i+1)^0.8 (vài pattern rất phổ biến).
- 10.000 giao dịch "nền" dài 32 item: 1–3 pattern + item ngẫu nhiên cho đủ 32.
- len05..len30: giao dịch thứ i lấy PREFIX của giao dịch nền thứ i, độ dài
  dao động quanh mức đích (±1 với len05, ±2 với các mức khác), trung bình đúng
  bằng mức đích. Vì là prefix của cùng giao dịch nền nên các mức lồng nhau.

Đây là dữ liệu có tương quan (correlated), không phải ngẫu nhiên độc lập.

In [ ]:
const SYN_N_TRANSACTIONS = 10_000
const SYN_N_ITEMS = 500
const SYN_MAX_LENGTH = 32
const SYN_TARGETS = (5, 10, 15, 20, 25, 30)

"Sinh 10.000 giao dịch nền (mỗi giao dịch 32 item, giữ thứ tự sinh)."
function synthetic_master(seed::Integer)
    pool_rng = PyRandom(seed)
    item_ids = collect(1:SYN_N_ITEMS)
    pyshuffle!(pool_rng, item_ids)
    patterns = [item_ids[i:i+3] for i in 1:4:SYN_N_ITEMS]
    for _ in 1:75
        push!(patterns, pysample_range!(pool_rng, 1, SYN_N_ITEMS + 1, 4))
    end
    length(patterns) == 200 || error("Số pattern phải là 200")

    weights = [1.0 / cpow(Float64(i + 1), 0.8) for i in 0:length(patterns)-1]
    cum_patterns = cumulative(weights)
    cum_npatterns = cumulative([25, 50, 25])   # cho n_patterns ∈ (1, 2, 3)

    rng = PyRandom(seed + 1)
    master = Vector{Vector{Int}}(undef, SYN_N_TRANSACTIONS)
    for t in 0:SYN_N_TRANSACTIONS-1
        transaction = Int[]
        seen = Set{Int}()
        n_patterns = (1, 2, 3)[pychoice_index!(rng, cum_npatterns)+1]
        first_pattern = t < 125 ? t : pychoice_index!(rng, cum_patterns)
        selected = [first_pattern]
        for _ in 1:n_patterns-1
            push!(selected, pychoice_index!(rng, cum_patterns))
        end
        for pid in selected, item in patterns[pid+1]
            if !(item in seen)
                push!(seen, item)
                push!(transaction, item)
            end
        end
        while length(transaction) < SYN_MAX_LENGTH
            item = pyrandint!(rng, 1, SYN_N_ITEMS)
            if !(item in seen)
                push!(seen, item)
                push!(transaction, item)
            end
        end
        master[t+1] = transaction
    end
    return master
end

"""
    gen_synthetic(outdir; seed = 2026) -> Vector{NamedTuple}

Ghi `len05.txt` … `len30.txt` và `synthetic_statistics.csv` vào `outdir`.
"""
function gen_synthetic(outdir::AbstractString; seed::Integer = 2026)
    mkpath(outdir)
    master = synthetic_master(seed)
    stats = NamedTuple[]
    previous = nothing
    for target in SYN_TARGETS
        offsets = target == 5 ? (-1, -1, 0, 1, 1) : (-2, -1, 0, 1, 2)
        per = SYN_N_TRANSACTIONS ÷ length(offsets)
        lengths = [target + o for o in offsets for _ in 1:per]
        pyshuffle!(PyRandom(seed + target), lengths)

        file = joinpath(outdir, "len$(lpad(target, 2, '0')).txt")
        current = Vector{Set{Int}}(undef, SYN_N_TRANSACTIONS)
        unique_items = Set{Int}()
        open(file, "w") do io
            for (i, len) in enumerate(lengths)
                items = sort(master[i][1:len])
                current[i] = Set(items)
                union!(unique_items, items)
                println(io, join(items, ' '))
            end
        end
        if previous !== nothing
            all(issubset(a, b) for (a, b) in zip(previous, current)) ||
                error("len$target không chứa mức trước (không lồng nhau)")
        end
        previous = current
        avg = sum(lengths) / SYN_N_TRANSACTIONS
        avg == target || error("Độ dài trung bình $avg ≠ $target")
        length(unique_items) == SYN_N_ITEMS || error("Không phủ đủ $SYN_N_ITEMS item")
        push!(stats, (dataset = basename(file), n_transactions = SYN_N_TRANSACTIONS,
            n_items = length(unique_items), avg_length = avg,
            min_length = minimum(lengths), max_length = maximum(lengths),
            density = avg / SYN_N_ITEMS, seed = seed, sha256 = sha256_file(file)))
    end
    header = ["dataset", "n_transactions", "n_items", "avg_length", "min_length",
        "max_length", "density", "seed", "sha256"]
    write_csv(joinpath(outdir, "synthetic_statistics.csv"), header,
        [[getfield(s, Symbol(h)) for h in header] for s in stats])
    return stats
end

### Chạy bước 4

In [ ]:
synthetic = gen_synthetic(SYNTHETIC_DIR)
for s in synthetic
    println(s.sha256 == expected[s.dataset] ? "[OK]   " : "[SAI]  ", rpad(s.dataset, 12),
            "avg=", s.avg_length, "  min=", s.min_length, "  max=", s.max_length)
end
@assert all(s -> s.sha256 == expected[s.dataset], synthetic) "Synthetic datasets không khớp SHA-256"
println()
println("Hoàn tất. Dữ liệu nằm trong ", DATA)